# Bluesky Sinner vs Alcaraz — Social Network & Sentiment Analysis

This notebook consolidates the full analysis pipeline for studying the Bluesky discourse around **Jannik Sinner** and **Carlos Alcaraz** during the **US Open 2025**.

### Pipeline
1. **Data Import & Preprocessing** — load the crawled posts, clean/tokenise text, generate a word cloud
2. **Social Network Analysis** — build interaction graphs, compute centralities, detect communities, plot network visualisations
3. **Social Sentiment Analysis** — NLP enrichment (RoBERTa sentiment + NRC/BERT emotions), plot sentiment distributions, trajectories, and community emotion profiles

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS & GLOBAL SETUP
# ═══════════════════════════════════════════════════════════════════════════════

import os
import re
import sys
import ast
import html
import json
import random
import subprocess
from typing import Any, Optional
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import infomap
import spacy
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from wordcloud import WordCloud

# Set custom HuggingFace cache directory
os.environ["HF_HOME"] = os.path.abspath(".huggingface_cache")

import torch
from tqdm import tqdm
from nrclex import NRCLex
from transformers import pipeline as hf_pipeline

# ── Reproducibility ──
random.seed(42)
np.random.seed(42)

# ── Matplotlib / Seaborn styling ──
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

# ── NLTK resources ──
for _resource in ['stopwords', 'punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'corpora/{_resource}' if _resource == 'stopwords' else f'tokenizers/{_resource}')
    except LookupError:
        nltk.download(_resource, quiet=True)

# ── spaCy model ──
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("spaCy model 'en_core_web_sm' not found, downloading...")
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp = spacy.load("en_core_web_sm")

print("All imports loaded successfully.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS & CONSTANTS  (from utils.py)
# ═══════════════════════════════════════════════════════════════════════════════

# US Open 2025 key match dates, used to annotate sentiment timelines.
US_OPEN_EVENTS = [
    ("2025-08-24", "US Open begins", "grey"),
    ("2025-08-25", "Alcaraz R1", "blue"),
    ("2025-08-26", "Sinner R1", "blue"),
    ("2025-08-27", "R2 matches", "blue"),
    ("2025-08-30", "R3 matches", "blue"),
    ("2025-09-01", "R4 matches", "blue"),
    ("2025-09-03", "Quarterfinals", "orange"),
    ("2025-09-05", "Semifinals", "red"),
    ("2025-09-07", "Final\n(Alcaraz wins)", "darkred"),
]

# US Open 2025 round windows, used to group median sentiment per round.
US_OPEN_ROUNDS = [
    {"label": "Pre-Tournament", "start": "2025-08-21", "end": "2025-08-23"},
    {"label": "US Open begins",  "start": "2025-08-24", "end": "2025-08-24"},
    {"label": "R1",              "start": "2025-08-25", "end": "2025-08-26"},
    {"label": "R2",              "start": "2025-08-27", "end": "2025-08-28"},
    {"label": "R3",              "start": "2025-08-29", "end": "2025-08-30"},
    {"label": "R4",              "start": "2025-08-31", "end": "2025-09-01"},
    {"label": "Quarterfinals",   "start": "2025-09-02", "end": "2025-09-03"},
    {"label": "Semifinals",      "start": "2025-09-04", "end": "2025-09-05"},
    {"label": "Final",           "start": "2025-09-06", "end": "2025-09-07"},
    {"label": "Post-Final",      "start": "2025-09-08", "end": "2025-09-09"},
]


def parse_list_col(val: Any) -> list:
    """Parse a column value stored as a string-encoded list back into a Python list."""
    if isinstance(val, list):
        return val
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        try:
            return json.loads(val)
        except (ValueError, TypeError):
            return []


def build_did_to_handle(df: pd.DataFrame) -> dict[str, str]:
    """Build a mapping from author DID to author handle using the posts DataFrame."""
    did_to_handle: dict[str, str] = {}
    for did, handle in zip(df['author_did'], df['author_handle']):
        if pd.notna(did) and pd.notna(handle):
            did_to_handle[did] = handle
    return did_to_handle


def save_plot_copies(filename: str, output_dir: str = "plots") -> None:
    """Save the current Matplotlib figure to output_dir and to the mirrored report directory."""
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches='tight')

    report_dir = output_dir.replace("plots", "report", 1)
    os.makedirs(report_dir, exist_ok=True)
    plt.savefig(os.path.join(report_dir, filename), dpi=300, bbox_inches='tight')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING FUNCTIONS  (from preprocessing.py)
# ═══════════════════════════════════════════════════════════════════════════════

tokenizer = TweetTokenizer()
custom_stopwords = set(stopwords.words('english')).union({'tennis', 'match', 'play', 'player', 'set'})


def clean_text(text: Optional[str]) -> str:
    """Normalise raw post text: unescape HTML entities, strip URLs and @handles, collapse whitespace."""
    if pd.isna(text):
        return ""
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def clean_text_bert(text: Optional[str]) -> str:
    """Light normalisation for BERT/RoBERTa: unescape HTML, replace handles with @user, replace URLs with http."""
    if pd.isna(text):
        return ""
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', 'http', text)
    text = re.sub(r'@\w+', '@user', text)
    return re.sub(r'\s+', ' ', text).strip()


def preprocess(text: Optional[str]) -> str:
    """Clean, tokenise and lemmatise post text into a space-joined string of meaningful tokens."""
    cleaned = clean_text(text)
    tokens = tokenizer.tokenize(cleaned.lower())

    doc = spacy.tokens.Doc(nlp.vocab, words=tokens)
    for name in ["tagger", "attribute_ruler", "lemmatizer"]:
        if name in nlp.pipe_names:
            doc = nlp.get_pipe(name)(doc)

    filtered_tokens = []
    for token in doc:
        t = token.text
        lemma = token.lemma_.lower()
        if lemma in custom_stopwords or t in custom_stopwords:
            continue
        if t.isalnum() or t.startswith('#') or not t.isascii():
            if len(lemma) > 1 or not t.isascii():
                filtered_tokens.append(lemma)

    return " ".join(filtered_tokens)


def load_data(filepath: str, parse_linked_entities: bool = False) -> tuple[Optional[pd.DataFrame], Optional[dict]]:
    """Load, parse dates/lists, and build DID->handle map for raw or processed datasets."""
    for directory in ("data", "plots", "report"):
        os.makedirs(directory, exist_ok=True)

    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"Error: {filepath} not found!")
        return None, None

    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', format='mixed')
    df = df.dropna(subset=['created_at'])

    list_columns = ['mentions', 'hashtags', 'links']
    if parse_linked_entities:
        list_columns.append('linked_entities')

    for col in list_columns:
        if col in df.columns:
            df[col] = df[col].apply(parse_list_col)

    return df, build_did_to_handle(df)


def prepare_dataset(raw_filepath: str, processed_filepath: str) -> tuple[Optional[pd.DataFrame], Optional[dict]]:
    """Return the NLP-enriched dataset and DID->handle map.

    Loads a cached processed file when available; otherwise runs the full NLP
    enrichment over the raw crawled posts.
    """
    if os.path.exists(processed_filepath):
        print(f"[INFO] Processed dataset found at {processed_filepath}. Loading it directly...")
        return load_data(processed_filepath, parse_linked_entities=True)

    df, did_to_handle = load_data(raw_filepath, parse_linked_entities=False)
    if df is None:
        return None, None

    df_processed = run_nlp_enrichment(df, output_filepath=processed_filepath)["df"]
    return df_processed, did_to_handle


def plot_community_wordcloud(df: pd.DataFrame, output_dir: str = "plots") -> None:
    """Generate and save a single word cloud for the entire community using preprocessed text."""
    if 'preprocessed_text' not in df.columns:
        print("Warning: 'preprocessed_text' column not found. Skipping word cloud.")
        return

    valid_texts = df['preprocessed_text'].dropna().astype(str).tolist()
    valid_texts = [t for t in valid_texts if t.strip() != ""]

    if not valid_texts:
        print("Warning: No words found for community word cloud. Skipping.")
        return

    community_text = " ".join(valid_texts)

    wordcloud_community = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        random_state=42,
        max_words=300
    ).generate(community_text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud_community, interpolation='bilinear')
    plt.axis('off')
    plt.title("Bluesky Community - Overall Word Cloud\n(US Open 2025)", fontsize=16, pad=15, weight='bold')
    plt.tight_layout()
    save_plot_copies("community_wordcloud.png", output_dir=output_dir)
    plt.show()
    plt.close()

---
## 1. Data Import & Preprocessing

Load the crawled Bluesky posts CSV, run text preprocessing, and visualise the overall word cloud.

In [ ]:
RAW_DATA_PATH = "data/sinner_alcaraz_posts.csv"
PROCESSED_DATA_PATH = "data/sinner_alcaraz_processed.csv"

df_processed, did_to_handle = prepare_dataset(RAW_DATA_PATH, PROCESSED_DATA_PATH)
if df_processed is None:
    raise RuntimeError("Failed to load dataset. Ensure the CSV files exist in data/.")

print(f"Dataset loaded: {len(df_processed)} posts, {df_processed.shape[1]} columns")
df_processed.head()

In [ ]:
# ── Word Cloud ──
plot_community_wordcloud(df_processed)

---
## 2. Social Network Analysis

Build undirected and directed interaction graphs from reply/mention relations, compute centrality measures, detect communities (Louvain & Infomap), and render network visualisations.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SOCIAL NETWORK ANALYSIS FUNCTIONS  (from social_network_analysis.py)
# ═══════════════════════════════════════════════════════════════════════════════

EMOTION_COLORS: dict[str, str] = {
    "anger":        "#e74c3c",
    "anticipation": "#e67e22",
    "disgust":      "#795548",
    "fear":         "#9b59b6",
    "joy":          "#f1c40f",
    "sadness":      "#3498db",
    "surprise":     "#1abc9c",
    "trust":        "#2ecc71",
    "neutral":      "#95a5a6",
}


def build_networks(df: pd.DataFrame, did_to_handle: dict) -> tuple[nx.Graph, nx.DiGraph]:
    """Build undirected and directed interaction graphs from reply and mention relations."""
    Gu = nx.Graph()
    Gd = nx.DiGraph()

    def add_edge(source: str, target: str, relationship: str) -> None:
        if source == target:
            return
        for graph in (Gu, Gd):
            if graph.has_edge(source, target):
                graph[source][target]['weight'] += 1
            else:
                graph.add_edge(source, target, weight=1, relationship=relationship)

    for _, row in df.iterrows():
        src_did = row['author_did']
        if pd.isna(src_did):
            continue
        source = did_to_handle.get(src_did, src_did)

        parent_did = row['reply_parent_did']
        if pd.notna(parent_did):
            add_edge(source, did_to_handle.get(parent_did, parent_did), "REPLY")

        for m in row['mentions']:
            m_did = m.get('did') if isinstance(m, dict) else m
            if m_did:
                add_edge(source, did_to_handle.get(m_did, m_did), "MENTION")

    return Gu, Gd


def calculate_centralities(Gu: nx.Graph, Gd: nx.DiGraph) -> dict:
    """Compute degree/closeness/betweenness centralities (both graphs) plus PageRank on Gd."""
    deg_cent = nx.degree_centrality(Gu)
    close_cent = nx.closeness_centrality(Gu)
    between_cent = nx.betweenness_centrality(Gu)

    in_deg_cent = nx.in_degree_centrality(Gd)
    out_deg_cent = nx.out_degree_centrality(Gd)
    close_cent_dir = nx.closeness_centrality(Gd)
    between_cent_dir = nx.betweenness_centrality(Gd)

    try:
        pagerank = nx.pagerank(Gd, weight='weight')
    except Exception:
        pagerank = {node: 0.0 for node in Gd.nodes()}

    nx.set_node_attributes(Gu, deg_cent, "degree_centrality")
    nx.set_node_attributes(Gu, close_cent, "closeness_centrality")
    nx.set_node_attributes(Gu, between_cent, "betweenness_centrality")

    return {
        "deg_cent": deg_cent,
        "close_cent": close_cent,
        "between_cent": between_cent,
        "in_deg_cent": in_deg_cent,
        "out_deg_cent": out_deg_cent,
        "close_cent_dir": close_cent_dir,
        "between_cent_dir": between_cent_dir,
        "pagerank": pagerank,
    }


def run_community_detection(Gu: nx.Graph, Gd: nx.DiGraph) -> dict:
    """Detect communities (Louvain on Gu, Infomap on Gd) and compute global network statistics."""
    if len(Gu.nodes()) <= 1:
        print("Graph too small for community detection / GCC calculation.")
        return {
            "communities": [list(Gu.nodes())],
            "modularity_score": 0.0,
            "node_to_community": {},
            "infomap_communities": [],
            "infomap_modularity": 0.0,
            "node_to_infomap": {},
            "gcc": Gu,
            "gcc_size": len(Gu.nodes()),
            "gcc_fraction": 1.0 if len(Gu.nodes()) > 0 else 0.0,
            "deg_assort_undir": 0.0,
            "deg_assort_dir": 0.0,
            "comm_assort_undir": 0.0,
        }

    communities = nx.community.louvain_communities(Gu)
    modularity_score = nx.community.modularity(Gu, communities)

    node_to_community = {node: i for i, comm in enumerate(communities) for node in comm}
    nx.set_node_attributes(Gu, node_to_community, "community")
    nx.set_node_attributes(Gd, node_to_community, "community")

    # Infomap operates on integer node IDs
    im = infomap.Infomap("--two-level --silent")
    node_to_id = {node: idx for idx, node in enumerate(Gd.nodes())}
    id_to_node = {idx: node for node, idx in node_to_id.items()}
    for u, v, data in Gd.edges(data=True):
        im.add_link(node_to_id[u], node_to_id[v], float(data.get('weight', 1.0)))
    im.run()

    node_to_infomap = {}
    infomap_communities_dict: dict[int, list] = {}
    for node_it in im.iterLeafNodes():
        orig_node = id_to_node.get(node_it.physicalId)
        if orig_node is not None:
            node_to_infomap[orig_node] = node_it.module_id
            infomap_communities_dict.setdefault(node_it.module_id, []).append(orig_node)

    for node in Gd.nodes():
        node_to_infomap.setdefault(node, -1)

    infomap_communities = [set(nodes) for nodes in infomap_communities_dict.values()]
    try:
        infomap_modularity = nx.community.modularity(Gd, infomap_communities)
    except Exception:
        infomap_modularity = 0.0

    nx.set_node_attributes(Gu, node_to_infomap, "community_infomap")
    nx.set_node_attributes(Gd, node_to_infomap, "community_infomap")

    density = nx.density(Gu)
    transitivity = nx.transitivity(Gu)
    avg_clustering = nx.average_clustering(Gu)

    components = sorted(nx.connected_components(Gu), key=len, reverse=True)
    gcc = Gu.subgraph(components[0])
    gcc_size = gcc.number_of_nodes()
    gcc_fraction = gcc_size / Gu.number_of_nodes()

    gcc_avg_path_length = gcc_diameter = gcc_radius = min_ecc = max_ecc = 0.0
    if gcc_size > 1:
        try:
            gcc_avg_path_length = nx.average_shortest_path_length(gcc)
        except Exception:
            gcc_avg_path_length = 0.0
        try:
            gcc_ecc = nx.eccentricity(gcc)
            gcc_diameter = nx.diameter(gcc, gcc_ecc)
            gcc_radius = nx.radius(gcc, gcc_ecc)
            min_ecc = min(gcc_ecc.values())
            max_ecc = max(gcc_ecc.values())
        except Exception:
            gcc_diameter = gcc_radius = min_ecc = max_ecc = 0.0

    print("\n" + "=" * 50)
    print("GLOBAL SOCIAL NETWORK STATISTICS")
    print("=" * 50)
    print(f"Network Density:                    {density:.6f}")
    print(f"Network Transitivity:               {transitivity:.6f}")
    print(f"Average Clustering Coefficient:     {avg_clustering:.6f}")
    print(f"GCC Size:                           {gcc_size} nodes ({gcc_fraction*100:.2f}% of graph)")
    print(f"GCC Average Shortest Path Length:   {gcc_avg_path_length:.4f}")
    print(f"GCC Diameter (Max Eccentricity):    {gcc_diameter:.1f}")
    print(f"GCC Radius (Min Eccentricity):      {gcc_radius:.1f}")
    print(f"Louvain Communities:                {len(communities)} (Modularity Q: {modularity_score:.4f})")
    print(f"Infomap Communities:                {len(infomap_communities)} (Modularity Q: {infomap_modularity:.4f})")
    print("=" * 50)

    try:
        os.makedirs("data", exist_ok=True)
        stats_df = pd.DataFrame({
            "Metric": [
                "Density", "Transitivity", "Average Clustering",
                "GCC Size", "GCC Fraction", "GCC Avg Path Length",
                "GCC Diameter", "GCC Radius", "GCC Min Eccentricity", "GCC Max Eccentricity",
                "Louvain Modularity", "Infomap Modularity",
            ],
            "Value": [
                density, transitivity, avg_clustering,
                gcc_size, gcc_fraction, gcc_avg_path_length,
                gcc_diameter, gcc_radius, min_ecc, max_ecc,
                modularity_score, infomap_modularity,
            ],
        })
        stats_df.to_csv("data/network_global_metrics.csv", index=False)
    except Exception as e:
        print(f"Error saving global metrics: {e}")

    try:
        deg_assort_undir = nx.degree_assortativity_coefficient(Gu)
    except Exception as e:
        deg_assort_undir = 0.0
        print("Error calculating undirected degree assortativity:", e)
    try:
        deg_assort_dir = nx.degree_assortativity_coefficient(Gd)
    except Exception as e:
        deg_assort_dir = 0.0
        print("Error calculating directed degree assortativity:", e)
    try:
        comm_assort_undir = nx.attribute_assortativity_coefficient(Gu, "community")
    except Exception as e:
        comm_assort_undir = 0.0
        print("Error calculating undirected community assortativity:", e)

    return {
        "louvain_communities": communities,
        "modularity_score": modularity_score,
        "node_to_louvain": node_to_community,
        "infomap_communities": infomap_communities,
        "infomap_modularity": infomap_modularity,
        "node_to_infomap": node_to_infomap,
        "gcc": gcc,
        "gcc_size": gcc_size,
        "gcc_fraction": gcc_fraction,
        "deg_assort_undir": deg_assort_undir,
        "deg_assort_dir": deg_assort_dir,
        "comm_assort_undir": comm_assort_undir,
    }


def save_initial_centrality_csv(
    Gu: nx.Graph,
    centralities: dict,
    comm_data: dict,
    filepath: str = "data/network_centrality_metrics.csv",
) -> pd.DataFrame:
    """Assemble per-node centrality and community metrics into a DataFrame and save it to CSV."""
    multi_node = len(Gu.nodes()) > 1
    node_to_louvain = comm_data["node_to_louvain"]
    node_to_infomap = comm_data.get("node_to_infomap", {})

    centrality_data = [{
        "user": node,
        "community": node_to_louvain.get(node, 0) if multi_node else 0,
        "community_infomap": node_to_infomap.get(node, -1) if multi_node else -1,
        "degree_centrality_undirected": centralities["deg_cent"].get(node, 0.0),
        "in_degree_centrality_directed": centralities["in_deg_cent"].get(node, 0.0),
        "out_degree_centrality_directed": centralities["out_deg_cent"].get(node, 0.0),
        "closeness_centrality_undirected": centralities["close_cent"].get(node, 0.0),
        "closeness_centrality_directed": centralities["close_cent_dir"].get(node, 0.0),
        "betweenness_centrality_undirected": centralities["between_cent"].get(node, 0.0),
        "betweenness_centrality_directed": centralities["between_cent_dir"].get(node, 0.0),
        "pagerank": centralities["pagerank"].get(node, 0.0),
    } for node in Gu.nodes()]

    df_cent = pd.DataFrame(centrality_data)
    df_cent.to_csv(filepath, index=False)
    return df_cent


def get_community_color_map(node_to_community: dict, cmap_name: str = "viridis") -> dict:
    """Map each community ID to a hex colour drawn from the named colormap."""
    unique_cids = sorted(set(node_to_community.values()))
    n = len(unique_cids)

    try:
        cmap = plt.colormaps.get_cmap(cmap_name)
    except AttributeError:
        cmap = plt.cm.get_cmap(cmap_name)

    is_qualitative = cmap_name.lower().startswith(('pastel', 'paired', 'accent', 'dark2', 'set', 'tab'))

    color_map = {}
    for idx, cid in enumerate(unique_cids):
        if is_qualitative:
            rgba = cmap(idx % cmap.N)
        else:
            rgba = cmap(idx / (n - 1) if n > 1 else 0.5)
        color_map[cid] = mcolors.to_hex(rgba)
    return color_map


def get_filtered_networks(Gu: nx.Graph, Gd: nx.DiGraph, min_component_size: int = 10) -> tuple[nx.Graph, nx.DiGraph]:
    """Return copies of Gu/Gd with connected components of size <= min_component_size removed."""
    Gu_filtered = Gu.copy()
    for component in list(nx.connected_components(Gu_filtered)):
        if len(component) <= min_component_size:
            Gu_filtered.remove_nodes_from(component)

    Gd_filtered = Gd.copy()
    for component in list(nx.weakly_connected_components(Gd_filtered)):
        if len(component) <= min_component_size:
            Gd_filtered.remove_nodes_from(component)

    return Gu_filtered, Gd_filtered


def _select_top_communities(
    graph,
    node_to_community: dict,
    df_processed: pd.DataFrame,
    top_k: int,
    sort_by: str,
) -> set:
    """Return the set of top-k community IDs within `graph`, ranked by post volume or node count."""
    if sort_by == "post_volume":
        author_community = {
            handle: node_to_community[handle]
            for handle in df_processed['author_handle']
            if handle in node_to_community and handle in graph.nodes()
        }
        comm_ids = df_processed['author_handle'].map(author_community).dropna()
        return set(comm_ids.value_counts().head(top_k).index.tolist())

    filtered = {n: c for n, c in node_to_community.items() if n in graph.nodes()}
    return {cid for cid, _ in Counter(filtered.values()).most_common(top_k)}


def plot_filtered_network_graph(
    Gu: nx.Graph,
    Gd: nx.DiGraph,
    df_cent: pd.DataFrame,
    comm_data: dict,
    centralities: dict,
    df_processed: pd.DataFrame,
    min_component_size: int = 10,
    output_dir: str = "plots/filtered",
    top_k: int = 5,
    sort_by: str = "post_volume",
    cmap_undirected: str = "tab20",
    cmap_directed: str = "tab20",
) -> tuple[nx.Graph, nx.DiGraph, dict, dict]:
    """Render network graphs restricted to large components and the top-k communities."""
    Gu_plot = Gu.copy()
    for component in list(nx.connected_components(Gu_plot)):
        if len(component) <= min_component_size:
            Gu_plot.remove_nodes_from(component)

    Gd_plot = Gd.copy()
    for component in list(nx.weakly_connected_components(Gd_plot)):
        if len(component) <= min_component_size:
            Gd_plot.remove_nodes_from(component)

    node_to_louvain = comm_data["node_to_louvain"]
    top_louvain = _select_top_communities(Gu_plot, node_to_louvain, df_processed, top_k, sort_by)
    Gu_plot.remove_nodes_from([n for n in list(Gu_plot.nodes()) if node_to_louvain.get(n) not in top_louvain])

    node_to_infomap = comm_data["node_to_infomap"]
    top_infomap = _select_top_communities(Gd_plot, node_to_infomap, df_processed, top_k, sort_by)
    Gd_plot.remove_nodes_from([n for n in list(Gd_plot.nodes()) if node_to_infomap.get(n) not in top_infomap])

    nodes_with_edges = [n for n, d in Gu.degree() if d > 0]
    pos = nx.spring_layout(Gu.subgraph(nodes_with_edges), k=0.3, iterations=60, seed=42)

    nodes_with_edges_dir = [n for n, d in Gd.degree() if d > 0]
    pos_dir = nx.spring_layout(Gd.subgraph(nodes_with_edges_dir), k=0.3, iterations=60, seed=42)

    plot_network_graphs(
        Gu_plot, Gd_plot, df_cent, comm_data, centralities,
        output_dir=output_dir, pos=pos, pos_dir=pos_dir,
        cmap_undirected=cmap_undirected, cmap_directed=cmap_directed,
    )

    return Gu_plot, Gd_plot, pos, pos_dir


def _get_user_dominant_emotion(df_processed: pd.DataFrame, backend: str) -> dict[str, str]:
    """Return a mapping of author_handle -> dominant emotion (mode across all their posts)."""
    col = "nrc_dominant_emotion" if backend == "nrc" else "bert_dominant_emotion"
    grouped = df_processed.groupby("author_handle")[col].agg(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "neutral"
    )
    return grouped.to_dict()


def _add_emotion_legend(ax: plt.Axes) -> None:
    """Attach a fixed colour legend for the 8 Plutchik emotions + neutral."""
    patches = [
        mpatches.Patch(color=color, label=emotion.capitalize())
        for emotion, color in EMOTION_COLORS.items()
    ]
    ax.legend(
        handles=patches,
        loc="lower left",
        fontsize=8,
        title="Dominant Emotion",
        title_fontsize=9,
        framealpha=0.85,
        ncol=2,
    )


def plot_network_graphs_by_emotion(
    Gu: nx.Graph,
    Gd: nx.DiGraph,
    df_cent: pd.DataFrame,
    centralities: dict,
    df_processed: pd.DataFrame,
    backend: str = "nrc",
    output_dir: str = "plots",
    pos: Optional[dict] = None,
    pos_dir: Optional[dict] = None,
) -> None:
    """Render undirected and directed network graphs with nodes coloured by dominant emotion."""
    deg_cent = centralities["deg_cent"]
    pagerank = centralities["pagerank"]
    user_emotion = _get_user_dominant_emotion(df_processed, backend)
    backend_label = backend.upper()

    nodes_in_relations = [n for n, d in Gu.degree() if d > 0]
    if not nodes_in_relations:
        print("Isolated graph / Not enough relationships to plot.")
        return

    subG = Gu.subgraph(nodes_in_relations)
    if pos is None:
        pos = nx.spring_layout(subG, k=0.3, iterations=60, seed=42)

    top_10_nodes = df_cent.sort_values(by="degree_centrality_undirected", ascending=False).head(10)['user'].tolist()
    labels_to_draw = {node: node for node in subG.nodes() if node in top_10_nodes}

    fig, ax = plt.subplots(figsize=(12, 12))
    node_colors = [EMOTION_COLORS.get(user_emotion.get(node, "neutral"), EMOTION_COLORS["neutral"]) for node in subG.nodes()]
    node_sizes = [50 + (deg_cent[node] * 1200) for node in subG.nodes()]

    nx.draw_networkx_edges(subG, pos, ax=ax, alpha=0.15, edge_color="grey")
    nx.draw_networkx_nodes(subG, pos, ax=ax, node_size=node_sizes, node_color=node_colors, alpha=0.9)
    nx.draw_networkx_labels(subG, pos, ax=ax, labels=labels_to_draw, font_size=9, font_weight="bold", font_color="#1e272c")
    ax.set_title(f"Undirected Social Network Graph — Dominant Emotion ({backend_label})\n(node size proportional to degree centrality)", pad=15)
    ax.axis("off")
    _add_emotion_legend(ax)
    plt.tight_layout()
    save_plot_copies(f"network_graph_emotion_{backend}.png", output_dir=output_dir)
    plt.show()
    plt.close(fig)

    nodes_in_relations_dir = [n for n, d in Gd.degree() if d > 0]
    fig2, ax2 = plt.subplots(figsize=(12, 12))
    if nodes_in_relations_dir:
        subG_dir = Gd.subgraph(nodes_in_relations_dir)
        if pos_dir is None:
            pos_dir = nx.spring_layout(subG_dir, k=0.3, iterations=60, seed=42)

        node_colors_dir = [EMOTION_COLORS.get(user_emotion.get(node, "neutral"), EMOTION_COLORS["neutral"]) for node in subG_dir.nodes()]
        node_sizes_dir = [50 + (pagerank.get(node, 0.0) * 18000) for node in subG_dir.nodes()]

        nx.draw_networkx_edges(subG_dir, pos_dir, ax=ax2, alpha=0.2, edge_color="grey",
                               arrows=True, arrowstyle='-|>', arrowsize=12,
                               connectionstyle="arc3,rad=0.1")
        nx.draw_networkx_nodes(subG_dir, pos_dir, ax=ax2, node_size=node_sizes_dir, node_color=node_colors_dir, alpha=0.9)

        top_10_nodes_dir = df_cent.sort_values(by="pagerank", ascending=False).head(10)['user'].tolist()
        labels_to_draw_dir = {node: node for node in subG_dir.nodes() if node in top_10_nodes_dir}
        nx.draw_networkx_labels(subG_dir, pos_dir, ax=ax2, labels=labels_to_draw_dir, font_size=9, font_weight="bold", font_color="#1e272c")
    else:
        ax2.text(0.5, 0.5, "Isolated graph / Not enough relationships", ha='center', va='center')

    ax2.set_title(f"Directed Social Network Graph — Dominant Emotion ({backend_label})\n(node size proportional to PageRank prestige)", pad=15)
    ax2.axis("off")
    _add_emotion_legend(ax2)
    plt.tight_layout()
    save_plot_copies(f"network_graph_directed_emotion_{backend}.png", output_dir=output_dir)
    plt.show()
    plt.close(fig2)


def plot_network_graphs(
    Gu: nx.Graph,
    Gd: nx.DiGraph,
    df_cent: pd.DataFrame,
    comm_data: dict,
    centralities: dict,
    output_dir: str = "plots",
    pos: Optional[dict] = None,
    pos_dir: Optional[dict] = None,
    cmap_undirected: str = "tab20",
    cmap_directed: str = "tab20",
) -> tuple[Optional[dict], Optional[dict]]:
    """Render the undirected (Louvain/degree) and directed (Infomap/PageRank) network figures."""
    deg_cent = centralities["deg_cent"]
    pagerank = centralities["pagerank"]
    node_to_community = comm_data["node_to_louvain"]
    modularity_score = comm_data["modularity_score"]

    nodes_in_relations = [n for n, d in Gu.degree() if d > 0]
    if not nodes_in_relations:
        print("Isolated graph / Not enough relationships to plot.")
        return pos, pos_dir

    subG = Gu.subgraph(nodes_in_relations)
    if pos is None:
        pos = nx.spring_layout(subG, k=0.3, iterations=60, seed=42)

    top_10_nodes = df_cent.sort_values(by="degree_centrality_undirected", ascending=False).head(10)['user'].tolist()
    labels_to_draw = {node: node for node in subG.nodes() if node in top_10_nodes}

    plt.figure(figsize=(12, 12))
    louvain_color_map = get_community_color_map(node_to_community, cmap_name=cmap_undirected)
    node_colors = [louvain_color_map.get(node_to_community.get(node, 0), "#bdc3c7") for node in subG.nodes()]
    node_sizes = [50 + (deg_cent[node] * 1200) for node in subG.nodes()]

    nx.draw_networkx_edges(subG, pos, alpha=0.15, edge_color="grey")
    nx.draw_networkx_nodes(subG, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9)
    nx.draw_networkx_labels(subG, pos, labels=labels_to_draw, font_size=9, font_weight="bold", font_color="#1e272c")
    plt.title(f"Undirected Social Network Graph (degree centrality & Louvain partitions)\n(Modularity Q: {modularity_score:.4f})", pad=15)
    plt.axis("off")
    plt.tight_layout()
    save_plot_copies("network_graph.png", output_dir=output_dir)
    plt.show()
    plt.close()

    plt.figure(figsize=(12, 12))
    nodes_in_relations_dir = [n for n, d in Gd.degree() if d > 0]
    infomap_modularity = comm_data.get("infomap_modularity", 0.0)
    if nodes_in_relations_dir:
        subG_dir = Gd.subgraph(nodes_in_relations_dir)
        if pos_dir is None:
            pos_dir = nx.spring_layout(subG_dir, k=0.3, iterations=60, seed=42)

        node_to_infomap = comm_data.get("node_to_infomap", {})
        infomap_color_map = get_community_color_map(node_to_infomap, cmap_name=cmap_directed)
        node_colors_dir = [infomap_color_map.get(node_to_infomap.get(node, 0), "#bdc3c7") for node in subG_dir.nodes()]
        node_sizes_dir = [50 + (pagerank.get(node, 0.0) * 18000) for node in subG_dir.nodes()]

        nx.draw_networkx_edges(subG_dir, pos_dir, alpha=0.2, edge_color="grey",
                               arrows=True, arrowstyle='-|>', arrowsize=12,
                               connectionstyle="arc3,rad=0.1")
        nx.draw_networkx_nodes(subG_dir, pos_dir, node_size=node_sizes_dir, node_color=node_colors_dir, alpha=0.9)

        top_10_nodes_dir = df_cent.sort_values(by="pagerank", ascending=False).head(10)['user'].tolist()
        labels_to_draw_dir = {node: node for node in subG_dir.nodes() if node in top_10_nodes_dir}
        nx.draw_networkx_labels(subG_dir, pos_dir, labels=labels_to_draw_dir, font_size=9, font_weight="bold", font_color="#1e272c")
    else:
        plt.text(0.5, 0.5, "Isolated graph / Not enough relationships", ha='center', va='center')

    plt.title(f"Directed Social Network Graph (PageRank prestige & Infomap partitions)\n(Projected Modularity Q: {infomap_modularity:.4f})", pad=15)
    plt.axis("off")
    plt.tight_layout()
    save_plot_copies("network_graph_directed.png", output_dir=output_dir)
    plt.show()
    plt.close()

    return pos, pos_dir

In [ ]:
# ── Build networks & compute centralities ──
Gu, Gd = build_networks(df_processed, did_to_handle)
if Gu.number_of_nodes() == 0:
    raise RuntimeError("Network has 0 nodes. SNA modeling cannot proceed.")

print(f"Undirected graph: {Gu.number_of_nodes()} nodes, {Gu.number_of_edges()} edges")
print(f"Directed graph:   {Gd.number_of_nodes()} nodes, {Gd.number_of_edges()} edges")

centralities = calculate_centralities(Gu, Gd)
comm_data = run_community_detection(Gu, Gd)
df_cent = save_initial_centrality_csv(Gu, centralities, comm_data)

Gu_filtered, Gd_filtered = get_filtered_networks(Gu, Gd)

In [ ]:
# ── Plot community-coloured network graphs (full) ──
pos, pos_dir = plot_network_graphs(Gu, Gd, df_cent, comm_data, centralities)

In [ ]:
# ── Plot filtered network graphs (top-5 communities, large components only) ──
Gu_plot, Gd_plot, pos_f, pos_dir_f = plot_filtered_network_graph(
    Gu, Gd, df_cent, comm_data, centralities, df_processed)

In [ ]:
# ── Plot emotion-coloured network graphs (NRC & BERT, full + filtered) ──
for backend in ("nrc", "bert"):
    print(f"\n--- Emotion-coloured graphs ({backend.upper()}) — Full ---")
    plot_network_graphs_by_emotion(Gu, Gd, df_cent, centralities, df_processed,
                                   backend=backend, pos=pos, pos_dir=pos_dir)
    print(f"--- Emotion-coloured graphs ({backend.upper()}) — Filtered ---")
    plot_network_graphs_by_emotion(Gu_plot, Gd_plot, df_cent, centralities, df_processed,
                                   backend=backend, output_dir="plots/filtered",
                                   pos=pos_f, pos_dir=pos_dir_f)

---
## 3. Social Sentiment Analysis

Analyse emotion profiles per community, overall sentiment distribution, sentiment trajectories over the tournament, and compare NRC vs BERT emotion backends.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SOCIAL SENTIMENT ANALYSIS FUNCTIONS  (from social_sentiment_analysis.py)
# ═══════════════════════════════════════════════════════════════════════════════

# The 8 primary NRC emotion categories.
NRC_EMOTIONS = [
    'fear', 'anger', 'anticipation', 'trust',
    'surprise', 'sadness', 'disgust', 'joy'
]

# Mapping from GoEmotions (28 fine-grained labels) to the 8 NRC/Plutchik categories.
GOEMOTIONS_TO_NRC = {
    "joy":          ["joy", "amusement", "excitement", "love", "pride", "relief"],
    "trust":        ["admiration", "approval", "gratitude", "caring"],
    "anticipation": ["curiosity", "desire", "optimism"],
    "surprise":     ["surprise", "realization", "confusion"],
    "fear":         ["fear", "nervousness"],
    "anger":        ["anger", "annoyance", "disapproval"],
    "disgust":      ["disgust"],
    "sadness":      ["sadness", "disappointment", "grief", "remorse", "embarrassment"],
}

POSITIVE_EMOTIONS = ["emotion_joy", "emotion_trust", "emotion_anticipation"]
NEGATIVE_EMOTIONS = ["emotion_anger", "emotion_disgust", "emotion_sadness", "emotion_fear"]
W_SENTIMENT = 0.50
W_EMOTION = 0.35
W_FREQUENCY = 0.15

SINNER_URI = "http://dbpedia.org/resource/Jannik_Sinner"
ALCARAZ_URI = "http://dbpedia.org/resource/Carlos_Alcaraz"

SINNER_KEYWORDS = {"sinner", "jannik"}
ALCARAZ_KEYWORDS = {"alcaraz", "carlos", "carlitos"}

# Lazily-initialised GoEmotions (RoBERTa) classifier for the BERT emotion backend.
_emotion_clf_bert = None


def _get_emotion_clf_bert():
    global _emotion_clf_bert
    if _emotion_clf_bert is None:
        device = 0 if torch.cuda.is_available() else -1
        print(f"[NLP] Initializing GoEmotions BERT model on device={device}...")
        _emotion_clf_bert = hf_pipeline(
            "text-classification",
            model="SamLowe/roberta-base-go_emotions",
            top_k=None,
            truncation=True,
            max_length=128,
            device=device,
        )
    return _emotion_clf_bert


def extract_ner(text: Optional[str]) -> list[tuple[str, str]]:
    """Extract unique (entity_text, label) pairs for PERSON/ORG/GPE/LOC entities via spaCy."""
    if pd.isna(text) or text == "":
        return []
    ents = []
    for ent in nlp(text).ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "LOC"]:
            ents.append((ent.text.strip().replace("\n", " "), ent.label_))
    return list(set(ents))


def link_entities_dbpedia(text: Optional[str], confidence: float = 0.5) -> list[dict]:
    """Link the two rival players to their DBpedia URIs using local keyword matching."""
    if not text or pd.isna(text) or str(text).strip() == "":
        return []

    text_lower = text.lower()
    entities = []
    if any(k in text_lower for k in SINNER_KEYWORDS):
        entities.append({"surface_form": "Jannik Sinner", "uri": SINNER_URI})
    if any(k in text_lower for k in ALCARAZ_KEYWORDS):
        entities.append({"surface_form": "Carlos Alcaraz", "uri": ALCARAZ_URI})
    return entities


def score_emotions(text: str) -> dict[str, float]:
    """Score a single text across the 8 NRC emotion categories."""
    try:
        if not text or pd.isna(text) or str(text).strip() == "":
            return {e: 0.0 for e in NRC_EMOTIONS}
    except (TypeError, ValueError):
        return {e: 0.0 for e in NRC_EMOTIONS}

    try:
        emotion_obj = NRCLex()
        emotion_obj.load_raw_text(str(text))
        freqs = emotion_obj.affect_frequencies
        return {e: freqs.get(e, 0.0) for e in NRC_EMOTIONS}
    except Exception:
        return {e: 0.0 for e in NRC_EMOTIONS}


def add_emotion_columns(df: pd.DataFrame, text_col: str = 'cleaned_text', backend: str = "nrc") -> pd.DataFrame:
    """Add the 8 emotion columns plus a 'dominant_emotion' column to a DataFrame."""
    if backend == "bert":
        clf = _get_emotion_clf_bert()
        batch_size = 128
        emotion_inputs = df[text_col].tolist()

        def emotion_generator():
            for t in emotion_inputs:
                yield t if (isinstance(t, str) and t.strip() != "") else " "

        print(f"[NLP] Running GPU parallelized inference with batch_size={batch_size}...")
        raw_results = clf(
            emotion_generator(),
            batch_size=batch_size,
            truncation=True,
            max_length=128,
        )

        emotion_rows = []
        dominant_emotions = []
        for res in tqdm(raw_results, total=len(emotion_inputs), desc="GoEmotions BERT"):
            scores = {d["label"]: d["score"] for d in res}
            nrc_scores = {
                nrc: float(sum(scores.get(g, 0.0) for g in go))
                for nrc, go in GOEMOTIONS_TO_NRC.items()
            }
            emotion_rows.append(nrc_scores)

            raw_max_label = max(scores, key=scores.get)
            if raw_max_label == "neutral":
                dominant_emotions.append("neutral")
            else:
                dominant_emotions.append(max(nrc_scores, key=nrc_scores.get))

        emotion_df = pd.DataFrame(emotion_rows, index=df.index)
        emotion_df = emotion_df[NRC_EMOTIONS]
    else:
        emotion_df = pd.DataFrame(df[text_col].apply(score_emotions).tolist(), index=df.index)

    emotion_df.columns = [f'emotion_{e}' for e in NRC_EMOTIONS]
    stale_cols = [c for c in emotion_df.columns if c in df.columns]
    if 'dominant_emotion' in df.columns:
        stale_cols.append('dominant_emotion')
    if stale_cols:
        df = df.drop(columns=stale_cols)
    df = pd.concat([df, emotion_df], axis=1)

    if backend == "bert":
        df['dominant_emotion'] = dominant_emotions
    else:
        emotion_cols = list(emotion_df.columns)
        df['dominant_emotion'] = df[emotion_cols].idxmax(axis=1).str.replace('emotion_', '')
        df.loc[df[emotion_cols].sum(axis=1) == 0, 'dominant_emotion'] = 'neutral'
    return df


def derive_player_sentiment_scores(df: pd.DataFrame) -> dict[str, list[float]]:
    """Bucket compound scores per player (Sinner / Alcaraz)."""
    sinner_scores: list[float] = []
    alcaraz_scores: list[float] = []
    for linked_ents, comp, text in zip(df['linked_entities'], df['sentiment_compound'], df['text']):
        uris = {ent['uri'] for ent in linked_ents if isinstance(ent, dict)}
        text_lower = str(text).lower()
        if SINNER_URI in uris or any(k in text_lower for k in SINNER_KEYWORDS):
            sinner_scores.append(comp)
        if ALCARAZ_URI in uris or any(k in text_lower for k in ALCARAZ_KEYWORDS):
            alcaraz_scores.append(comp)
    return {"sinner_scores": sinner_scores, "alcaraz_scores": alcaraz_scores}


def run_nlp_enrichment(df: pd.DataFrame, output_filepath: str = "data/sinner_alcaraz_processed.csv", emotion_backend: str = "both") -> dict:
    """Enrich crawled posts with cleaned text, RoBERTa sentiment, emotions, NER and entity links."""
    df = df.copy()
    total_posts = len(df)
    print(f"[NLP] Starting BERT/RoBERTa NLP enrichment for {total_posts} posts...")

    print("[NLP] Step 1/6: Cleaning and preprocessing post text...")
    df['cleaned_text'] = df['text'].apply(clean_text)
    df['preprocessed_text'] = df['text'].apply(preprocess)
    print("[NLP] Step 1/6 completed.")

    print("[NLP] Step 2/6: Scoring RoBERTa sentiments on GPU...")
    device = 0 if torch.cuda.is_available() else -1
    print(f"[NLP] Initializing CardiffNLP RoBERTa model on device={device}...")
    classifier = hf_pipeline(
        "sentiment-analysis",
        model="cardiffnlp/twitter-roberta-base-sentiment-latest",
        device=device,
        top_k=None
    )

    bert_inputs = df['text'].apply(clean_text_bert).tolist()
    batch_size = 128
    sentiment_categories = []
    sentiment_compounds = []

    print(f"[NLP] Running GPU parallelized inference with batch_size={batch_size}...")

    def input_generator():
        for t in bert_inputs:
            yield t if (isinstance(t, str) and t.strip() != "") else " "

    results = classifier(input_generator(), batch_size=batch_size, truncation=True, max_length=512)

    for res in tqdm(results, total=len(bert_inputs), desc="RoBERTa Sentiment"):
        scores_dict = {d['label']: d['score'] for d in res}

        if 'LABEL_0' in scores_dict:
            p_neg = scores_dict.get('LABEL_0', 0.0)
            p_neu = scores_dict.get('LABEL_1', 0.0)
            p_pos = scores_dict.get('LABEL_2', 0.0)

            max_label = max(scores_dict, key=scores_dict.get)
            if max_label == 'LABEL_0':
                cat = 'negative'
            elif max_label == 'LABEL_2':
                cat = 'positive'
            else:
                cat = 'neutral'
        else:
            p_neg = scores_dict.get('negative', 0.0)
            p_neu = scores_dict.get('neutral', 0.0)
            p_pos = scores_dict.get('positive', 0.0)

            cat = max(scores_dict, key=scores_dict.get)

        comp = p_pos - p_neg

        sentiment_categories.append(cat)
        sentiment_compounds.append(comp)

    df['sentiment_category'] = sentiment_categories
    df['sentiment_compound'] = sentiment_compounds
    print("[NLP] Step 2/6 completed.")

    if emotion_backend == "nrc":
        print("[NLP] Step 3/6: Running NRC Emotion Lexicon analysis...")
        df = add_emotion_columns(df, text_col='cleaned_text', backend='nrc')
        print("[NLP] Step 3/6 completed.")
    elif emotion_backend == "bert":
        print("[NLP] Step 3/6: Running GoEmotions BERT emotion analysis...")
        df = add_emotion_columns(df, text_col='cleaned_text', backend='bert')
        for e in NRC_EMOTIONS:
            df[f'bert_emotion_{e}'] = df[f'emotion_{e}']
        df['bert_dominant_emotion'] = df['dominant_emotion']
        print("[NLP] Step 3/6 completed.")
    else:
        print("[NLP] Step 3/6: Running NRC Emotion Lexicon analysis...")
        df = add_emotion_columns(df, text_col='cleaned_text', backend='nrc')
        for e in NRC_EMOTIONS:
            df[f'nrc_emotion_{e}'] = df[f'emotion_{e}']
        df['nrc_dominant_emotion'] = df['dominant_emotion']
        print("[NLP] Step 3/6 completed.")

        print("[NLP] Step 4/6: Running GoEmotions BERT emotion analysis...")
        df = add_emotion_columns(df, text_col='cleaned_text', backend='bert')
        for e in NRC_EMOTIONS:
            df[f'bert_emotion_{e}'] = df[f'emotion_{e}']
        df['bert_dominant_emotion'] = df['dominant_emotion']
        print("[NLP] Step 4/6 completed.")

    print("[NLP] Step 5/6: Extracting Named Entities (spaCy NER)...")
    df['entities'] = df['cleaned_text'].apply(extract_ner)
    print("[NLP] Step 5/6 completed.")

    print("[NLP] Step 6/6: Linking entities to DBpedia resources...")
    df['linked_entities'] = df['cleaned_text'].apply(link_entities_dbpedia)
    print("[NLP] Step 6/6 completed.")

    scores = derive_player_sentiment_scores(df)

    df.to_csv(output_filepath, index=False)
    print(f"[NLP] Saved processed dataset to {output_filepath}")
    return {"df": df, **scores}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SENTIMENT PLOT FUNCTIONS  (from social_sentiment_analysis.py)
# ═══════════════════════════════════════════════════════════════════════════════

def plot_community_emotion_profiles(
    df: pd.DataFrame,
    Gu,
    node_to_community: dict,
    output_dir: str = "plots",
    title_suffix: str = "",
    top_k: int = 5,
    sort_by: str = "post_volume",
    cmap_name: str = "tab20",
    backend: str = "nrc",
) -> None:
    """Plot average emotion profiles for the top-k communities of the filtered graph Gu."""
    author_community = {}
    for handle in df['author_handle']:
        if handle in node_to_community and handle in Gu.nodes():
            author_community[handle] = node_to_community[handle]

    df = df.copy()
    df['community_id'] = df['author_handle'].map(author_community)
    df_with_comm = df.dropna(subset=['community_id']).copy()

    if sort_by == "post_volume":
        top_comms = df_with_comm['community_id'].value_counts().head(top_k).index.tolist()
    else:
        filtered = {n: c for n, c in node_to_community.items() if n in Gu.nodes()}
        top_comms = [cid for cid, _ in Counter(filtered.values()).most_common(top_k)]

    if backend == "bert":
        col_prefix = 'bert_emotion_'
    else:
        col_prefix = 'nrc_emotion_' if 'nrc_emotion_fear' in df.columns else 'emotion_'

    emotion_cols = [f'{col_prefix}{e}' for e in NRC_EMOTIONS]
    records = []
    custom_palette = {}
    color_map = get_community_color_map(node_to_community, cmap_name=cmap_name)

    for cid in top_comms:
        comm_posts = df_with_comm[df_with_comm['community_id'] == cid]
        if len(comm_posts) == 0:
            continue
        label = f"Comm {int(cid)} (N={len(comm_posts)})"
        custom_palette[label] = color_map.get(cid)
        avg_scores = comm_posts[emotion_cols].mean()
        for col in emotion_cols:
            records.append({"Community": label, "Emotion": col.replace(col_prefix, ''), "Score": avg_scores[col]})

    df_plot = pd.DataFrame(records)
    if df_plot.empty:
        return

    subtitle = "GoEmotions (BERT)" if backend == "bert" else "NRC Emotion Lexicon"
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df_plot, x="Emotion", y="Score", hue="Community", palette=custom_palette)
    plt.title(f"Average Emotion Profiles per Community{title_suffix}\n({subtitle})", fontsize=14, pad=15)
    plt.xlabel("Emotion Category")
    plt.ylabel("Mean Normalized Score")
    plt.legend(title="Community")
    plt.tight_layout()

    if "Louvain" in title_suffix:
        algo = "Louvain"
    elif "Infomap" in title_suffix:
        algo = "Infomap"
    else:
        algo = title_suffix.replace(' ', '_').replace('(', '').replace(')', '')
        if algo.startswith("_"):
            algo = algo[1:]

    filename = f"community_emotion_{algo}_{backend}.png"
    save_plot_copies(filename, output_dir=output_dir)
    plt.show()
    plt.close()


def plot_emotion_backend_comparison(df: pd.DataFrame, output_dir: str = "plots") -> None:
    """Plot a grouped bar chart comparing dominant-emotion distributions of both backends."""
    if 'nrc_dominant_emotion' not in df.columns or 'bert_dominant_emotion' not in df.columns:
        print("Warning: 'nrc_dominant_emotion'/'bert_dominant_emotion' columns not found. Skipping plot.")
        return

    emotion_index = NRC_EMOTIONS + ["neutral"]
    nrc_dist = df['nrc_dominant_emotion'].value_counts(normalize=True).reindex(emotion_index, fill_value=0.0)
    bert_dist = df['bert_dominant_emotion'].value_counts(normalize=True).reindex(emotion_index, fill_value=0.0)

    records = []
    for emotion in emotion_index:
        records.append({"Emotion": emotion, "Share of posts": float(nrc_dist[emotion]), "Backend": "NRC"})
        records.append({"Emotion": emotion, "Share of posts": float(bert_dist[emotion]), "Backend": "BERT"})
    df_plot = pd.DataFrame(records)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=df_plot, x="Emotion", y="Share of posts", hue="Backend")
    plt.title("Dominant Emotion Distribution: NRC vs GoEmotions (BERT)", fontsize=14, pad=15)
    plt.xlabel("Emotion Category")
    plt.ylabel("Share of posts")
    plt.legend(title="Backend")
    plt.tight_layout()

    save_plot_copies("emotion_backend_comparison.png", output_dir=output_dir)
    plt.show()
    plt.close()


def plot_sentiment_distribution(df: pd.DataFrame, output_dir: str = "plots") -> None:
    """Plot a donut chart showing the overall distribution of sentiment categories."""
    if 'sentiment_category' not in df.columns:
        print("Warning: 'sentiment_category' column not found in DataFrame. Skipping plot.")
        return

    counts = df['sentiment_category'].value_counts()
    categories = counts.index.tolist()
    sizes = counts.values.tolist()

    colors_map = {
        'positive': '#2ec4b6',
        'neutral': '#a0aec0',
        'negative': '#e63946'
    }
    colors = [colors_map.get(cat, '#cbd5e0') for cat in categories]

    plt.figure(figsize=(7, 7))
    wedges, texts, autotexts = plt.pie(
        sizes,
        labels=categories,
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        pctdistance=0.75,
        textprops=dict(color="black", fontsize=12, weight="bold")
    )

    centre_circle = plt.Circle((0,0), 0.55, fc='white')
    fig = plt.gcf()
    fig.gca().add_artist(centre_circle)

    for autotext in autotexts:
        autotext.set_color('white')

    plt.title("Overall Sentiment Distribution\n(RoBERTa Sentiment Classifier)", fontsize=14, pad=20, weight="bold")
    plt.tight_layout()
    save_plot_copies("sentiment_distribution.png", output_dir=output_dir)
    plt.show()
    plt.close()


def plot_sentiment_over_time(df: pd.DataFrame, output_dir: str = "plots") -> None:
    """Plot daily average sentiment compound scores for Sinner vs. Alcaraz over time."""
    if 'created_at' not in df.columns or 'sentiment_compound' not in df.columns or 'linked_entities' not in df.columns:
        print("Warning: Required columns for sentiment over time not found. Skipping plot.")
        return

    df = df.copy()
    df['date'] = pd.to_datetime(df['created_at'], format='mixed').dt.date

    sinner_records = []
    alcaraz_records = []

    for _, row in df.iterrows():
        linked_ents = row['linked_entities']
        if isinstance(linked_ents, str):
            try:
                linked_ents = ast.literal_eval(linked_ents)
            except Exception:
                linked_ents = []

        uris = {ent['uri'] for ent in linked_ents if isinstance(ent, dict)}
        text_lower = str(row['text']).lower()

        is_sinner = SINNER_URI in uris or any(k in text_lower for k in SINNER_KEYWORDS)
        is_alcaraz = ALCARAZ_URI in uris or any(k in text_lower for k in ALCARAZ_KEYWORDS)

        if is_sinner:
            sinner_records.append({"date": row['date'], "sentiment": row['sentiment_compound']})
        if is_alcaraz:
            alcaraz_records.append({"date": row['date'], "sentiment": row['sentiment_compound']})

    df_sinner = pd.DataFrame(sinner_records)
    df_alcaraz = pd.DataFrame(alcaraz_records)

    sinner_trend = df_sinner.groupby('date')['sentiment'].mean() if not df_sinner.empty else pd.Series()
    alcaraz_trend = df_alcaraz.groupby('date')['sentiment'].mean() if not df_alcaraz.empty else pd.Series()

    sinner_trend = sinner_trend.sort_index()
    alcaraz_trend = alcaraz_trend.sort_index()

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Post Volume on secondary Y-axis
    ax2 = ax1.twinx()

    sinner_vol = df_sinner.groupby('date')['sentiment'].count() if not df_sinner.empty else pd.Series()
    alcaraz_vol = df_alcaraz.groupby('date')['sentiment'].count() if not df_alcaraz.empty else pd.Series()

    sinner_vol = sinner_vol.sort_index()
    alcaraz_vol = alcaraz_vol.sort_index()

    if not sinner_vol.empty:
        ax2.fill_between(sinner_vol.index, sinner_vol.values, alpha=0.12, color='#f39c12', label='Sinner Post Volume')
    if not alcaraz_vol.empty:
        ax2.fill_between(alcaraz_vol.index, alcaraz_vol.values, alpha=0.12, color='#00b4d8', label='Alcaraz Post Volume')

    ax2.set_ylabel("Daily Post Volume (Shaded)", color='gray', fontsize=11, labelpad=10)
    ax2.tick_params(axis='y', labelcolor='gray')
    ax2.grid(False)

    # Average Sentiment on primary Y-axis
    if not sinner_trend.empty:
        ax1.plot(sinner_trend.index, sinner_trend.values, marker='o', linewidth=2.5, color='#d35400', label='Jannik Sinner Sentiment')
    if not alcaraz_trend.empty:
        ax1.plot(alcaraz_trend.index, alcaraz_trend.values, marker='s', linewidth=2.5, color='#023e8a', label='Carlos Alcaraz Sentiment')

    ax1.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)

    # Annotate US Open events
    ax1.relim()
    ax1.autoscale_view()
    ymin, ymax = ax1.get_ylim()
    text_y = ymax - (ymax - ymin) * 0.08

    for date_str, label, color in US_OPEN_EVENTS:
        try:
            event_date = pd.to_datetime(date_str).date()
            ax1.axvline(event_date, color=color, linestyle=':', alpha=0.5, linewidth=1.2)
            ax1.text(
                event_date,
                text_y,
                f"  {label}",
                rotation=90,
                verticalalignment='top',
                horizontalalignment='left',
                fontsize=9,
                color=color,
                weight='semibold',
                alpha=0.8
            )
        except Exception as e:
            print(f"Warning: Could not annotate event {label} at {date_str}: {e}")

    plt.title("Sentiment Trajectory & Post Volume Over Time (US Open 2025)\n(Daily Avg RoBERTa Sentiment vs. Post Volume)", fontsize=14, pad=15, weight="bold")
    ax1.set_xlabel("Date", fontsize=11, labelpad=10)
    ax1.set_ylabel("Average Sentiment Compound Score (Lines)", fontsize=11, labelpad=10)
    ax1.grid(True, linestyle=':', alpha=0.6)

    # Combine legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

    fig.autofmt_xdate()
    plt.tight_layout()

    save_plot_copies("sentiment_over_time.png", output_dir=output_dir)
    plt.show()
    plt.close()

In [ ]:
# ── Community Emotion Profiles (Louvain + Infomap × NRC + BERT) ──
print("[NLP] Generating community emotion profiles...")

plot_community_emotion_profiles(df_processed, Gu_filtered, comm_data["node_to_louvain"],
                                title_suffix=" (Louvain - Filtered)", backend="nrc")
plot_community_emotion_profiles(df_processed, Gu_filtered, comm_data["node_to_louvain"],
                                title_suffix=" (Louvain - Filtered)", backend="bert")
plot_community_emotion_profiles(df_processed, Gd_filtered, comm_data["node_to_infomap"],
                                title_suffix=" (Infomap - Filtered)", backend="nrc")
plot_community_emotion_profiles(df_processed, Gd_filtered, comm_data["node_to_infomap"],
                                title_suffix=" (Infomap - Filtered)", backend="bert")

In [ ]:
# ── Sentiment Distribution, Trajectory & Backend Comparison ──
print("[NLP] Generating sentiment visualization plots...")

plot_sentiment_distribution(df_processed)
plot_sentiment_over_time(df_processed)
plot_emotion_backend_comparison(df_processed)

print("[NLP] All plots successfully generated and saved to plots/")

---
## Summary

This notebook executed the complete Sinner vs Alcaraz analysis pipeline:

1. **Preprocessing** — Loaded and cleaned the Bluesky posts dataset, generated a word cloud of the community discourse.
2. **Social Network Analysis** — Built undirected/directed interaction graphs, computed centrality measures (degree, closeness, betweenness, PageRank), detected communities via Louvain and Infomap, and rendered full + filtered network graphs coloured by community and dominant emotion.
3. **Social Sentiment Analysis** — Visualised community emotion profiles (NRC & BERT backends, both Louvain and Infomap partitions), overall sentiment distribution, Sinner vs Alcaraz sentiment trajectories over the tournament, and NRC vs BERT backend comparison.

All plots are saved to `plots/` and mirrored to `report/`.